# Validated parcelwise DBM analysis

This notebook refuses legacy or partial outputs. Set RUN_ID to a corrected run that has passed validate_cohort_outputs.py; the subject manifest must contain exactly 137 Healthy and 130 Unhealthy subjects. Results use frozen parcel IDs, not column numbers.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts" / "analysis" / "parcel_statistics.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "scripts" / "analysis"))
from parcel_statistics import FEATURE_LOCATIONS, analyse_feature

RUN_ID = "REPLACE_WITH_VALIDATED_RUN_ID"
RUN_ROOT = PROJECT_ROOT / "outputs" / "runs" / RUN_ID
SUBJECTS = PROJECT_ROOT / "data" / "test_cohort_warps_available.txt"
DATABASE = PROJECT_ROOT / "data" / "database_finale_labels_corrette.csv"
STATISTICS_DIR = RUN_ROOT / "statistics"

if RUN_ID.startswith("REPLACE_"):
    raise ValueError("Set RUN_ID to the corrected, cohort-validated run ID first")
if not (RUN_ROOT / "cohort_validation.json").is_file():
    raise FileNotFoundError(
        f"Validate the completed run before inference: {RUN_ROOT / 'cohort_validation.json'}"
    )

The analysis below runs vectorized Welch tests with BH-FDR, Hedges' *g*, 95% confidence intervals, an age/sex/education-adjusted model, and leave-one-subject-out sensitivity. Weighted degree is interpreted as relative graph centrality; direct mean and median log-Jacobian summaries are analysed separately as deformation endpoints.

In [ ]:
results = {}
sensitivity = {}

for feature in FEATURE_LOCATIONS:
    feature_results, feature_sensitivity = analyse_feature(
        RUN_ROOT, SUBJECTS, DATABASE, feature
    )
    results[feature] = feature_results
    sensitivity[feature] = feature_sensitivity

summary = pd.DataFrame([
    {
        "feature": feature,
        "welch_bh_fdr_05": int(frame["significant_fdr_05"].sum()),
        "covariate_glm_bh_fdr_05": int(frame["glm_significant_fdr_05"].sum()),
        "n_parcels": len(frame),
    }
    for feature, frame in results.items()
]).set_index("feature")
summary

In [ ]:
# Parcel-ID overlap across representations; effects unique to one endpoint need investigation.
significant_sets = {
    feature: set(frame.loc[frame["significant_fdr_05"], "parcel_id"])
    for feature, frame in results.items()
}
overlap = pd.DataFrame(
    {
        right: {
            left: len(significant_sets[left] & significant_sets[right])
            for left in significant_sets
        }
        for right in significant_sets
    }
)
overlap

In [ ]:
# Inspect the strongest validated effects with anatomical parcel IDs.
feature = "weighted_degree_inv1pW"
results[feature].sort_values("p_fdr_bh").loc[
    :, [
        "parcel_id", "unhealthy_minus_healthy", "difference_ci_low",
        "difference_ci_high", "hedges_g", "p_fdr_bh",
        "glm_beta_unhealthy", "glm_p_fdr_bh",
    ]
].head(25)

In [ ]:
# Save one clean table per endpoint; rerunning replaces files instead of appending rows.
STATISTICS_DIR.mkdir(parents=True, exist_ok=True)
for feature, frame in results.items():
    frame.to_csv(STATISTICS_DIR / f"{feature}_parcel_statistics.csv", index=False)
    sensitivity[feature].to_csv(
        STATISTICS_DIR / f"{feature}_leave_one_out.csv", index=False
    )
summary.to_csv(STATISTICS_DIR / "feature_summary.csv")
overlap.to_csv(STATISTICS_DIR / "significant_parcel_overlap.csv")
STATISTICS_DIR

Permutation/max-T confirmation is deliberately deferred until the corrected 267-subject batch is complete and the preliminary endpoint comparison has been reviewed.